# Laboratorul 9 -> Procesarea textelor cu ajutorul LLMs

In Melk, in urma unor inundatii produse de Dunare, o parte din versurile unor poezii (aflate in volumele de poezii din biblioteca) s-au degradat. Folositi un LLM pentru a completa versurile lipsa (avand in vedere ca primul vers din fiecare strofa s-a pastrat intact).

Rezolvarea abordata: se foloseste un LLM pre-antrenat (pe texte generale) si se analizeaza influenta parametrilor (inclusiv a tokenizer-ului) asupra calitatii textului generat

In [33]:
import numpy as np
import prelucrarea_datelor
from prelucrarea_datelor import extrage_strofe

### Prelucrarea datelor

In [34]:
cale_folder = "poetry"

poezii = extrage_strofe(cale_folder)

print(poezii)

[{'fisier': 'EyeshinebyPaulCameronBrown30504.xml', 'titlu': 'THE INTRUDER', 'strofe': [{'primul_vers': 'The colouring of spacious flowers rove delicious to the eye .', 'restul': ['The road above the harbour fickle , carousing in its tendency', 'to pull too gray by sky enamelled water .', 'The tropical foliage , still and languorous , to my touch .', 'Each particle of sunlight dangling as if hoisted from', 'a perfumed ledge .', 'Newly mown grass in streaks , browns serpent-like across', 'the path .', 'Low erogenous puffs of dust are swathed by passing feet .', 'Near by , bushes wear the foliage of streaked mud as a mantle', 'might cottonwool at Christmas .']}]}, {'fisier': 'EyeshinebyPaulCameronBrown30504.xml', 'titlu': 'THE BAY OF CORTES', 'strofe': [{'primul_vers': 'A speck of a fisherman dots the horizon .', 'restul': ['His craft a barque in loneliness across the sea .', 'Dolphins inveigh the richness of the depths ,', 'persuade latitudes to drift about their wake .', 'Pelicans sour 

Alegerea modelului

In [35]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import os
from huggingface_hub import login

token = os.environ.get("TOKEN_HUGGING_FACE_AI9")
if token:
    login(token)

# alegerea modelului
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# incarcarea modelului
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=False,
    local_files_only=True
)

print("The model was succesfully loaded.")

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 403.02it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


The model was succesfully loaded.


In [36]:
# se alege temperatura de 0.7 pentru a avea o performanta cat mai mare, dar sa nu inventeze (ex pt temp = 0.8)
def restaureaza_versuri(primele_versuri, temp = 0.7):
    # instructiunile initiale
    '''
    prompt = "You are an expert in poetry. Complete the poem, knowing only the first verse for each stanza. "

    # parcurgem primele versuri ale fiecarei strofe din poezia curenta
    index = 1
    for vers in primele_versuri:
        prompt = prompt +f"\nVerse 1 of stanza {index}: {vers}"
        index += 1
'''

    message = ""
    for i, vers in enumerate(primele_versuri):
        message += f"Stanza {i+1}, Line 1: {vers}\n[...rest of the stanza is missing...]\n\n"

    # system -> defineste "personalitatea"
    # user -> defineste cerinta efectiva
    messages = [
    {"role": "system", "content": "You are an expert in poetry. Restore the missing lines of the provided stanzas. Do not add new stanzas. Provide only the completed text of the poem. Keep in mind that the stanzas have to make sense and rhyme."},
    {"role": "user", "content": f"Complete these stanzas knowing only the first lines:\n\n{message}"}
    ]

    # transforma in formatul cu care este obisnuit
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # acum avem promptul pregatit pentru poezia curenta -> mai trebuie doar tokenizat (ca sa-l inteleaga modelul)
    input = tokenizer(prompt, return_tensors="pt").to(model.device)

    # se trimit atat id-urile din input, cat si attention mask
    output = model.generate(**input,max_new_tokens=200,temperature=temp,do_sample=True,repetition_penalty=1.1)

    # acum trebuie sa transformam output-ul in ceva citibil si pentru oameni
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

    # genereaza doar raspunsul modelului
    doar_strofele = tokenizer.decode(output[0][input.input_ids.shape[-1]:], skip_special_tokens=True)

    print(generated_text)
    return doar_strofele


poezie_test = [poezie for poezie in poezii if poezie['fisier'] == "FactsinJinglesbyWinifredSackvilleStoner48044.xml" and poezie['titlu'] == "PETER VISITS AN EPISCOPAL CHURCH"][0]

# de test -> se alege pe o poezie propriu-zisa (spre exemplu, prima)
versuri = [[strofa['primul_vers'],strofa['restul']] for strofa in poezie_test['strofe']]

primele_versuri = [vers[0] for vers in versuri]

doar_strofele = restaureaza_versuri(primele_versuri)


system
You are an expert in poetry. Restore the missing lines of the provided stanzas. Do not add new stanzas. Provide only the completed text of the poem. Keep in mind that the stanzas have to make sense and rhyme.
user
Complete these stanzas knowing only the first lines:

Stanza 1, Line 1: When Peter who was a country jake
[...rest of the stanza is missing...]

Stanza 2, Line 1: And after service when his Ma
[...rest of the stanza is missing...]


assistant
When Peter who was a country jake,
He'd dreamt up stories under moonlight's glow.

The countryside he knew so well,
With fields of wheat and cows galloping by.
His heart yearned for adventure vast,
Where wild horses ran free on green pastures high.

And after service when his Ma
Would call him home from every walk,
Her voice like music through the night,
A tender reminder of his dear old town.


### BLEU - Bilangual Evaluation Understudy

In [39]:
import nltk
from nltk.translate.bleu_score import corpus_bleu
from nltk.tokenize import word_tokenize

strof = poezie_test['strofe']

predicted = []
for strofa in strof:
    s = ""
    s+= strofa['primul_vers']
    for vers in strofa['restul']:
        s+=vers

    cuv = word_tokenize(s)
    predicted.append([cuv])

print(predicted)
print()

generated = []
for strofa_gen in doar_strofele.split("\n\n"):
    cuv = word_tokenize(strofa_gen)
    generated.append(cuv)

print(generated)


# se echilibreaza nr de strofe
diferenta = len(predicted) - len(generated)

if diferenta > 0:
    for _ in range(diferenta):
        generated.append([])
elif diferenta < 0:
    for _ in range(-diferenta):
        predicted.append([[]])

score = corpus_bleu(predicted, generated)
print()
print(score)

[[['When', 'Peter', 'who', 'was', 'a', 'country', 'jakeA', 'visit', 'to', 'a', 'church', 'did', 'makeHe', 'sat', 'with', 'pleased', 'look', 'on', 'his', 'faceAs', 'if', 'indeed', 'in', 'Heaven', "'s", 'place', '.']], [['And', 'after', 'service', 'when', 'his', 'MaPraised', 'him', 'aloud', 'to', 'his', 'kind', 'PaHe', 'said', ',', '“', 'Of', 'course', 'I', 'sat', 'quite', 'stillAnd', 'watched', 'the', 'preacher', "'s", 'wives', 'so', 'illAll', 'dressed', 'in', 'nighties', ',', 'though', 'their', 'hairWas', 'primped', 'and', 'curled', 'as', 'for', 'a', 'fair', '.', '”']]]

[['When', 'Peter', 'who', 'was', 'a', 'country', 'jake', ',', 'He', "'d", 'dreamt', 'up', 'stories', 'under', 'moonlight', "'s", 'glow', '.'], ['The', 'countryside', 'he', 'knew', 'so', 'well', ',', 'With', 'fields', 'of', 'wheat', 'and', 'cows', 'galloping', 'by', '.', 'His', 'heart', 'yearned', 'for', 'adventure', 'vast', ',', 'Where', 'wild', 'horses', 'ran', 'free', 'on', 'green', 'pastures', 'high', '.'], ['And', 